# **Importing Libraries**

In [15]:
import re
import fitz  
import numpy as np
import easyocr
import unicodedata
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
import os
from groq import Groq

# **English Text Processing**

In [16]:
ENGLISH_PDF = "dostor in english.pdf"

print("=" * 50)
print("ENGLISH PDF — OCR EXTRACTION (EasyOCR)")
print("=" * 50)

reader = easyocr.Reader(['en'], gpu=True)
print("✅ EasyOCR model loaded")

doc = fitz.open(ENGLISH_PDF)
print(f"Total pages: {len(doc)}")

raw_text_en = ""
for i, page in enumerate(doc):
    # Render page to image at 300 DPI
    mat = fitz.Matrix(300 / 72, 300 / 72)
    pix = page.get_pixmap(matrix=mat)
    
    # Convert to numpy array (EasyOCR accepts numpy directly)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(
        pix.height, pix.width, pix.n
    )
    
    # If image has alpha channel (4 channels), drop it
    if pix.n == 4:
        img = img[:, :, :3]
    
    # detail=0 returns plain text list, detail=1 returns bounding boxes too
    results = reader.readtext(img, detail=0, paragraph=True)
    page_text = "\n".join(results)
    raw_text_en += page_text + "\n"
    print(f"  [EN] Processed page {i + 1}/{len(doc)}")

doc.close()
print(f"\nRaw English text length : {len(raw_text_en):,} characters")
print("---preview ---")
print(raw_text_en[7787:9500])

ENGLISH PDF — OCR EXTRACTION (EasyOCR)
✅ EasyOCR model loaded
Total pages: 124
  [EN] Processed page 1/124
  [EN] Processed page 2/124
  [EN] Processed page 3/124
  [EN] Processed page 4/124
  [EN] Processed page 5/124
  [EN] Processed page 6/124
  [EN] Processed page 7/124
  [EN] Processed page 8/124
  [EN] Processed page 9/124
  [EN] Processed page 10/124
  [EN] Processed page 11/124
  [EN] Processed page 12/124
  [EN] Processed page 13/124
  [EN] Processed page 14/124
  [EN] Processed page 15/124
  [EN] Processed page 16/124
  [EN] Processed page 17/124
  [EN] Processed page 18/124
  [EN] Processed page 19/124
  [EN] Processed page 20/124
  [EN] Processed page 21/124
  [EN] Processed page 22/124
  [EN] Processed page 23/124
  [EN] Processed page 24/124
  [EN] Processed page 25/124
  [EN] Processed page 26/124
  [EN] Processed page 27/124
  [EN] Processed page 28/124
  [EN] Processed page 29/124
  [EN] Processed page 30/124
  [EN] Processed page 31/124
  [EN] Processed page 32/124
  

In [17]:
# After your OCR loop finishes
with open("extracted_text_en.txt", "w", encoding="utf-8") as f:
    f.write(raw_text_en)
print("✅ Text saved to extracted_text_en.txt")

✅ Text saved to extracted_text_en.txt


In [4]:
# ============================================================
# CELL 3 — English Text Cleaning
# ============================================================

def clean_ocr_english(text: str) -> str:
    """Fix common OCR errors and normalise whitespace."""
    # Common OCR mis-reads of "Article"
    text = re.sub(
        r'\bArtic1e\b|\bArticie\b|\bARTIC1E\b|\bArt1cle\b',
        'Article', text, flags=re.IGNORECASE
    )
    # Collapse all whitespace to single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

clean_text_en = clean_ocr_english(raw_text_en)

print("=" * 50)
print("ENGLISH — AFTER CLEANING")
print("=" * 50)
print(f"Cleaned text length : {len(clean_text_en):,} characters")
print("---preview ---")
print(clean_text_en[7787:9500])

ENGLISH — AFTER CLEANING
Cleaned text length : 138,894 characters
---preview ---
 Article (1) The Arab Republic of Egypt is a sovereign, united and indivisible State, where no part may be given up, having a democratic republican system that is based on citizenship and rule of law: The Egyptian people are part of the Arab nation seeking to enhance its integration and unity. Egypt is part of the Islamic world, belongs to the African continent; cherishes its Asian dimension, and contributes to building human civilization. Article (2) Islam is the official religion of the State, its official language is Arabic. The principles of Islamic Sharia are the main source of legislation: Article (3) The principles of Christian and Jewish canons of Egyptian Christians and Jews are the main source of legislations that regulate their respective personal status, religious affairs, and selection of their religious leaders: Article (4) Sovereignty belongs only to the people, who shall exercise and protec

In [5]:
# ============================================================
# CELL 4 — English Article Chunking —
# ============================================================

def split_articles_english(text: str) -> list[dict]:
    """
    Handles ONLY this form: Article (1)
    """
    pattern = r'(Article\s*\(\s*\d+\s*\))'

    parts = re.split(pattern, text, flags=re.IGNORECASE)

    articles = []
    for i in range(1, len(parts), 2):
        header = parts[i].strip()
        body   = parts[i + 1].strip() if i + 1 < len(parts) else ""

        if len(body) < 20:
            continue

        num_match = re.search(r'\d+', header)
        num = int(num_match.group()) if num_match else -1

        articles.append({
            "article_id" : f"en_{num}",
            "article_num": num,
            "header"     : header,
            "text"       : f"{header} {body}",
            "lang"       : "en",
        })

    # Deduplicate — keep longest
    seen: dict[int, dict] = {}
    for art in articles:
        n = art["article_num"]
        if n not in seen or len(art["text"]) > len(seen[n]["text"]):
            seen[n] = art

    return sorted(seen.values(), key=lambda x: x["article_num"])


articles_en = split_articles_english(clean_text_en)

print("=" * 50)
print("ENGLISH — ARTICLE CHUNKING RESULTS")
print("=" * 50)
print(f"Total unique articles extracted : {len(articles_en)}")
print("--- First 5 articles ---")
for a in articles_en[:3]:
    print(a['text'])

ENGLISH — ARTICLE CHUNKING RESULTS
Total unique articles extracted : 254
--- First 5 articles ---
Article (1) The Arab Republic of Egypt is a sovereign, united and indivisible State, where no part may be given up, having a democratic republican system that is based on citizenship and rule of law: The Egyptian people are part of the Arab nation seeking to enhance its integration and unity. Egypt is part of the Islamic world, belongs to the African continent; cherishes its Asian dimension, and contributes to building human civilization.
Article (2) Islam is the official religion of the State, its official language is Arabic. The principles of Islamic Sharia are the main source of legislation:
Article (3) The principles of Christian and Jewish canons of Egyptian Christians and Jews are the main source of legislations that regulate their respective personal status, religious affairs, and selection of their religious leaders:


# **Arabic Text Processing**

In [6]:
# ============================================================
# UNIFIED CELL: Generic Native PDF Arabic Legal Parser
# ============================================================

_AR_INDIC = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

# Regex to handle kerning issues and RTL numbers (e.g., "م ا د ة ( 1 )")
_PAT_WITH_PAREN = re.compile(
    r"م\s?ا\s?د\s?ة\s*[\(\)]\s*"
    r"((?:[٠-٩]|\d)(?:[\n\s]*(?:[٠-٩]|\d))*)"
    r"\s*[\(\)]"
)
_PAT_NO_OPEN_PAREN = re.compile(r"مادة(\d+)\n\)")


def extract_text(pdf_path: str) -> str:
    """Extracts raw text from a native PDF."""
    doc = fitz.open(pdf_path)
    raw_text = "\n".join([page.get_text() for page in doc])
    doc.close()
    return raw_text


def clean_text(text: str) -> str:
    """Generically cleans PDF artifacts by targeting encoding root-causes."""
    text = unicodedata.normalize("NFC", text)
    
    # 1. Strip invisible bidi markers & Tashkeel
    for ch in '\u202a\u202b\u202c\u202d\u202e\u200e\u200f\u2066\u2067\u2068\u2069':
        text = text.replace(ch, '')
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text) 
    
    # ---------------------------------------------------------
    # ROOT FIX 1: Prefix Ligature Reversals
    # Fixes the engine reading "الإ" backwards as "اإل"
    # ---------------------------------------------------------
    text = text.replace("اإل", "الإ") 
    text = text.replace("األ", "الأ") 
    text = text.replace("اآل", "الآ") 
    
    # ---------------------------------------------------------
    # ROOT FIX 2: Word-Boundary Ligature Reversals ("لا" -> "ال")
    # ---------------------------------------------------------
    text = re.sub(r'\bال\b', 'لا', text)
    text = re.sub(r'\bوال\b', 'ولا', text)
    text = re.sub(r'\bإال\b', 'إلا', text)
    text = re.sub(r'\bأال\b', 'ألا', text)
    text = re.sub(r'\bفال\b', 'فلا', text)
    
    # ---------------------------------------------------------
    # ROOT FIX 3: Orphaned Letter De-fragmentation
    # Merges isolated letters back into their words (e.g., ال م حكمة -> المحكمة)
    # ---------------------------------------------------------
    text = re.sub(r'\b([بتثجحخدذرزسشصضطظعغفقكلمنهي])\s+', r'\1', text)
    
    # ---------------------------------------------------------
    # ROOT FIX 4: Generic Punctuation Spacing
    # ---------------------------------------------------------
    text = re.sub(r'\s+([،\.؛:؟!])', r'\1', text) # Remove space before punctuation
    text = re.sub(r'([،\.؛:؟!])(?=[^\s\d])', r'\1 ', text) # Add space after punctuation
    
    # 5. Clean excess whitespace
    return re.sub(r' {2,}', ' ', text).strip()


def _normalize_num(raw: str) -> int:
    """Handles RTL reversed digits extracted by the PDF engine."""
    d = "".join(raw.split())
    if re.fullmatch(r"[٠-٩]+", d):
        d = d[::-1] 
    return int(d.translate(_AR_INDIC))


def _normalize_body(body: str) -> str:
    """Fixes physical line breaks and orphaned trailing letters."""
    body = re.sub(r"([\u0600-\u06FF])\n([ىا])(?=[\n ،.،؟!]|$)", r"\1\2", body)
    body = re.sub(r"(?<!\n)\n(?!\n)", " ", body)
    return re.sub(r" +", " ", body).strip()


def chunk_articles(text: str, noise_phrases: list[str] = None) -> list[dict]:
    """Slices the text into structured dictionaries by article."""
    noise_phrases = noise_phrases or []
    seen: dict[int, int] = {}

    for m in _PAT_WITH_PAREN.finditer(text):
        try:
            n = _normalize_num(m.group(1))
            if n not in seen: seen[n] = m.start()
        except ValueError: continue

    for m in _PAT_NO_OPEN_PAREN.finditer(text):
        try:
            n = int(m.group(1))
            if n not in seen: seen[n] = m.start()
        except ValueError: continue

    headers = sorted((offset, num) for num, offset in seen.items())
    articles = []

    for i, (start, num) in enumerate(headers):
        end = headers[i + 1][0] if i + 1 < len(headers) else len(text)
        raw_body = text[start:end]
        
        raw_body = re.sub(r"^م\s?ا\s?د\s?ة[\s\S]{0,20}?\)\s*", "", raw_body).strip()
        if len(raw_body) < 10: continue
            
        body = _normalize_body(raw_body)
        
        # Strip whatever header noise the user passed in
        for noise in noise_phrases:
            body = body.replace(noise, '')
            
        articles.append({
            "article_id" : f"ar_{num}",
            "article_num": num,
            "header"     : f"مادة ({num})",
            "text"       : "\u200f" + f"مادة ({num})\n" + body.strip(),
            "lang"       : "ar",
        })

    articles.sort(key=lambda a: a["article_num"])
    return articles


def parse(pdf_path: str, noise_phrases: list[str] = None) -> list[dict]:
    """Executes the extraction, cleaning, and chunking pipeline."""
    print(f"📄 Extracting: {pdf_path} ...")
    raw_text = extract_text(pdf_path)
    
    print("🧹 Cleaning PDF artifacts and Arabic Ligatures...")
    clean_txt = clean_text(raw_text)
    
    print("✂️ Chunking into structural articles...")
    articles = chunk_articles(clean_txt, noise_phrases)
    
    print("\n" + "=" * 50)
    print("📊 PARSING REPORT")
    print("=" * 50)
    
    if articles:
        print(f"Total unique 'مادة' extracted: {len(articles)}")
        
        min_n = articles[0]["article_num"]
        max_n = articles[-1]["article_num"]
        found = {a["article_num"] for a in articles}
        missing = [n for n in range(min_n, max_n + 1) if n not in found]
        
        if missing:
            print(f"⚠️ Missing articles between {min_n} and {max_n}:\n   {missing}")
        else:
            print(f"✅ Sequence complete! All articles from {min_n} to {max_n} found.")
    else:
        print("❌ No articles found in document.")
        
    return articles


# ============================================================
# EXECUTION
# ============================================================

PDF_FILE = "dostor in arabic.pdf"

# Specific headers to ignore for this document
custom_noise_phrases = ['دستور جمهورية مصر العربية', 'ديباجة وثيقة الدستور']

# Run the pipeline
articles_ar = parse(PDF_FILE, noise_phrases=custom_noise_phrases)

# Print a preview
print("\n--- PREVIEW ---")
for a in articles_ar[:3]:
    print(f"\n[{a['article_id']}]\n{a['text']}")

📄 Extracting: dostor in arabic.pdf ...
🧹 Cleaning PDF artifacts and Arabic Ligatures...
✂️ Chunking into structural articles...

📊 PARSING REPORT
Total unique 'مادة' extracted: 254
✅ Sequence complete! All articles from 1 to 254 found.

--- PREVIEW ---

[ar_1]
‏مادة (1)
، جمهورية مصر العربية دولة ذات سيادة، موحدة لا تقبل التجزئة ولا ينزل عن شئ منها، نظامها جمهورى ديمقراطى، يقوم على أساس. المواطنة وسيادة القانون الشعب المصرى جزء من الأمة العربية يعمل على تكاملها ووح دتها، وم صر جزء من العالم الإسالمى، تنتمى، إلى القارة الإفريقية وتعتز بامتدادها الآسيوى، وتسهم فى. بناء الحضارة الإنسانية

[ar_2]
‏مادة (2)
الإسالم دين الدولة، واللغة العربية لغتها الرسمية، ومبادئ الشريعة. الإسالمية المصدر الرئيسى للتشريع

[ar_3]
‏مادة (3)
مبادئ شر ائع المصريين من المسيحيين واليهود المصدر الرئيسى، للتشريعات المنظ مة ألحوالهم الشخصية وشئونهم الدينية، واختيار. قياداتهم الروحية


# **Embeddings**

In [ ]:
# ============================================================
# CELL 8 — Multilingual Embeddings
# ============================================================

EMBED_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

all_articles = articles_en + articles_ar
documents    = [a["text"] for a in all_articles]
metadatas    = [{
    "article_id" : a["article_id"],
    "article_num": a["article_num"],
    "lang"       : a["lang"],
} for a in all_articles]

print(f"\nEncoding {len(documents)}")

embeddings = embed_model.encode(
    documents,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True,
)

print(f"✅ Embeddings shape : {embeddings.shape}")


Encoding 508


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Embeddings shape : (508, 384)


In [8]:
# ============================================================
# CELL 9 — FAISS Vector Indexing (Crash-Proof)
# ============================================================

# 1. Convert your existing 'embeddings' variable into a strict float32 numpy array
# This is required by FAISS and prevents memory access violations
safe_embeddings = np.array(embeddings, dtype=np.float32)

# 2. Get the dimension size from your embeddings (e.g., 1024 or 768)
dimension = safe_embeddings.shape[1]

# 3. Set up FAISS for Cosine Similarity
# To get cosine similarity in FAISS, we normalize the vectors and use Inner Product (IP)
faiss.normalize_L2(safe_embeddings)
index = faiss.IndexFlatIP(dimension)

# 4. Add the vectors to the FAISS index
index.add(safe_embeddings)

print(f"✅ FAISS Index created successfully!")
print(f"✅ Stored {index.ntotal} chunks in FAISS.")
print(f"   English articles : {len([m for m in metadatas if m.get('lang') == 'en'])}")
print(f"   Arabic  articles : {len([m for m in metadatas if m.get('lang') == 'ar'])}")
print("=" * 50)

✅ FAISS Index created successfully!
✅ Stored 508 chunks in FAISS.
   English articles : 254
   Arabic  articles : 254


# **Re-ranking**

In [9]:
# ============================================================
# CELL 10 — CrossEncoder Reranker
# ============================================================
RERANKER_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker = CrossEncoder(RERANKER_MODEL)

print(f"✅ Reranker loaded : {RERANKER_MODEL}")

✅ Reranker loaded : cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


In [10]:
def retrieve_rerank(
    query: str,
    top_k: int = 10,
    final_k: int = 5,
    lang_filter: str | None = None,
) -> list[tuple]:
    """
    Retrieve candidates from FAISS, apply manual metadata filtering, 
    and rerank the final top_k with CrossEncoder.
    """
    # 1. Embed and format the query for FAISS (must be float32 and normalized)
    query_embedding = embed_model.encode([query]) # Ensure this matches your model variable name
    query_embedding = np.array(query_embedding, dtype=np.float32)
    faiss.normalize_L2(query_embedding)
    
    # 2. Determine how many vectors to pull
    # If we are filtering by language, pull extra to account for discarded wrong-language docs
    search_k = top_k * 5 if lang_filter else top_k
    
    # 3. Search FAISS
    distances, indices = index.search(query_embedding, search_k)
    
    candidate_docs = []
    candidate_metas = []
    
    # 4. Map FAISS indices back to your data and apply the manual language filter
    for idx in indices[0]:
        if idx == -1: # FAISS returns -1 if it runs out of documents
            continue
            
        meta = metadatas[idx]
        
        # Apply the language filter manually
        if lang_filter and meta.get("lang") != lang_filter:
            continue
            
        # If it passes the filter, add it to our candidates
        candidate_docs.append(documents[idx])
        candidate_metas.append(meta)
        
        # Stop looping once we have exactly 'top_k' valid candidates
        if len(candidate_docs) == top_k:
            break

    # If no documents match, return empty
    if not candidate_docs:
        return []

    # 5. Rerank with CrossEncoder
    scores = reranker.predict([(query, doc) for doc in candidate_docs])
    
    # 6. Sort and return
    ranked = sorted(zip(candidate_docs, candidate_metas, scores), key=lambda x: x[2], reverse=True)
    
    return ranked[:final_k]

# **Answer Generation**

In [11]:
# ============================================================
# CELL 11 — Dual LLM Loading (Groq API: Qwen + Llama)
# ============================================================
print("=" * 50)
print("DUAL LLM — GROQ API (FREE TIER)")
print("=" * 50)

# Set your API key first
os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL_CONFIGS = {
    "qwen": {
        "name": "qwen/qwen3-32b",   # or qwen/qwen-2.5-7b-instruct
        "description": "Strong multilingual (Arabic + English), best for RAG reasoning",
    },
    "llama": {
        "name": "llama-3.3-70b-versatile",
        "description": "Best reasoning + summarization, very strong general model",
    },
}

_active_key = [None]


def get_model(key: str):
    """No loading needed — Groq is API-based (stateless)."""
    if key not in MODEL_CONFIGS:
        raise ValueError(f"Unknown model key: {key}")

    _active_key[0] = key
    return MODEL_CONFIGS[key]["name"]


def generate(key: str, messages, temperature=0.2, max_tokens=1024):
    """
    Unified inference function for both models
    """
    model_name = get_model(key)

    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content


print("\n✅ Groq models ready (no VRAM needed)")
print("   Available models:", list(MODEL_CONFIGS.keys()))

DUAL LLM — GROQ API (FREE TIER)

✅ Groq models ready (no VRAM needed)
   Available models: ['qwen', 'llama']


In [12]:
def build_prompt(query: str, context: str, lang: str, model_key: str):

    if model_key == "llama":
        if lang == "ar":
            system_msg = (
                "أنت مساعد قانوني دقيق. "
                "استخدم فقط السياق المقدم لك. "
                "لا تضف أي معلومات خارج السياق. "
                "أجب في فقرة واحدة قصيرة فقط مع ذكر أرقام المواد."
            )
            user_msg = (
                f"السياق:\n{context}\n\n"
                f"السؤال: {query}\n\n"
                "أجب باختصار وبالاعتماد فقط على السياق:"
            )
        else:
            system_msg = (
                "You are a strict legal assistant. "
                "Use ONLY the provided context. "
                "Do NOT hallucinate or add external knowledge. "
                "Answer in ONE short paragraph and cite article numbers."
            )
            user_msg = (
                f"Context:\n{context}\n\n"
                f"Question: {query}\n\n"
                "Answer strictly using the context:"
            )
        return [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]

    # ── QWEN ──────────────────────────────────────────────────
    if lang == "ar":
        system_msg = (
            "أنت مساعد قانوني. القواعد الصارمة:\n"
            "1. استخدم السياق المقدم فقط.\n"
            "2. لا يُسمح باستخدام المعرفة الخارجية.\n"
            "3. فقرة واحدة فقط.\n"
            "4. اذكر أرقام المواد داخل النص.\n"
            "5. لا تتجاوز 100 كلمة."
        )
        user_msg = (
            f"السياق:\n{context}\n\n"
            f"السؤال: {query}\n\n"
            "/no_think\n"           # ← MUST be in user message for Qwen
            "أجب باختصار وبالاعتماد فقط على السياق:"
        )
    else:
        system_msg = (
            "You are a legal assistant. Strict rules:\n"
            "1. Use ONLY the given context.\n"
            "2. No external knowledge allowed.\n"
            "3. One paragraph only.\n"
            "4. Cite article numbers inline.\n"
            "5. Max 100 words."
        )
        user_msg = (
            f"Context:\n{context}\n\n"
            f"Question: {query}\n\n"
            "/no_think\n"           # ← MUST be in user message for Qwen
            "Answer strictly based on context:"
        )

    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]


def generate_answer(
    query: str,
    lang: str = "auto",
    model_key: str = "qwen",
) -> tuple[str, list[str]]:

    import re

    if lang == "auto":
        arabic_chars = len(re.findall(r'[\u0600-\u06FF]', query))
        lang = "ar" if arabic_chars > len(query) * 0.3 else "en"

    print(f"\n[Language: {lang} | Model: {model_key}]")

    results = retrieve_rerank(query, lang_filter=lang)
    if not results:
        return "No relevant articles found.", []

    context, citations = "", []
    for doc, meta, score in results:
        context += doc + "\n\n"
        citations.append(meta["article_id"])
        print(f"  Retrieved: {meta['article_id']} (score={score:.3f})")

    messages = build_prompt(query, context, lang, model_key)
    model_name = MODEL_CONFIGS[model_key]["name"]

    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.1,
        max_tokens=300,     # ← increased: 150 was cutting off mid-think
        top_p=0.9,
    )

    answer = response.choices[0].message.content

    # ── Safety net: strip any leftover <think> blocks ──
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
    answer = answer.split("\n\n")[0].strip()

    return answer, citations

In [13]:
# ============================================================
# CELL 13 — Demo Queries with model switching
# ============================================================

# ── English with Qwen ─────────────────────────────────────
print("=" * 50)
print("ENGLISH — Qwen")
print("=" * 50)
q_en = "What does the constitution say about freedom of expression?"
answer, cites = generate_answer(q_en, lang="en", model_key="qwen")
print(f"\nQ: {q_en}")
print(f"A: {answer}")
print(f"Citations: {cites}")

# ── English with Llama ────────────────────────────────────
print("\n" + "=" * 50)
print("ENGLISH — Llama")
print("=" * 50)
answer, cites = generate_answer(q_en, lang="en", model_key="llama")
print(f"\nQ: {q_en}")
print(f"A: {answer}")
print(f"Citations: {cites}")

# ── Arabic with Qwen ──────────────────────────────────────
print("\n" + "=" * 50)
print("ARABIC — Qwen")
print("=" * 50)
q_ar = "ما الذي يقوله الدستور عن حرية التعبير؟"
answer, cites = generate_answer(q_ar, lang="ar", model_key="qwen")
print(f"\nQ: {q_ar}")
print(f"A: {answer}")
print(f"Citations: {cites}")

# ── Arabic with Llama ─────────────────────────────────────
print("\n" + "=" * 50)
print("ARABIC — Llama")
print("=" * 50)
answer, cites = generate_answer(q_ar, lang="ar", model_key="llama")
print(f"\nQ: {q_ar}")
print(f"A: {answer}")
print(f"Citations: {cites}")

ENGLISH — Qwen

[Language: en | Model: qwen]
  Retrieved: en_65 (score=0.943)
  Retrieved: en_57 (score=-3.758)
  Retrieved: en_4 (score=-4.094)
  Retrieved: en_98 (score=-4.095)
  Retrieved: en_68 (score=-4.156)

Q: What does the constitution say about freedom of expression?
A: The Constitution guarantees freedom of thought and opinion, allowing every person the right to express their opinion verbally, in writing, through imagery, or by any other means of expression and publication, as stated in Article (65).
Citations: ['en_65', 'en_57', 'en_4', 'en_98', 'en_68']

ENGLISH — Llama

[Language: en | Model: llama]
  Retrieved: en_65 (score=0.943)
  Retrieved: en_57 (score=-3.758)
  Retrieved: en_4 (score=-4.094)
  Retrieved: en_98 (score=-4.095)
  Retrieved: en_68 (score=-4.156)

Q: What does the constitution say about freedom of expression?
A: According to Article (65), freedom of thought and opinion is guaranteed, and every person has the right to express their opinion verbally, in wri